In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import torch.autograd as autograd

device = torch.device("cuda:3")

In [ ]:
# Data preparation
class RecDataset(Dataset):
    def __init__(self, data, vocab_dict) -> None:
        super().__init__()
        self.data = data
        self.dict = vocab_dict

    def __getitem__(self, idx):
        seq = self.data[idx]
        seq = ['<start>'] + seq.replace('', ' ').strip().split() + ['<eos>']
        padding = ['<pad>'] * (27 - len(seq))
        seq = seq + padding
        seq_ids = [self.dict[i] for i in seq]
        seq_ids = torch.as_tensor(seq_ids)
        return seq_ids
    
    def __len__(self):
        return len(self.data)
    

vocab_df = pd.read_csv('vocab.dict', sep=' ',header=None)

vocab_df.columns = ['word', 'index']
vocab = dict(zip(vocab_df['word'], vocab_df['index']))
vocab_inv = {v: k for k, v in vocab.items()}

vocab_dict = np.array(vocab_df)
seq2idx = {k:v for k, v in vocab_dict}
idx2seq = {v:k for k, v in vocab_dict}

data = pd.read_csv('/data/home/weng501/Users/kjh/dataset/ic50_pred_onlymy.csv')
# data = data[data['ic50'] <= 40]
data = data[data['sequence'].apply(lambda x: len(x) <= 25)]
train_data, test_data = train_test_split(np.array(data['sequence']), test_size=0.2, random_state=42)


In [ ]:
#weight_matrix
def calculate_amino_acid_frequency(peptides):
    amino_acid_num = {'A':0, 'G':0, 'L':0, 'P':0, 'E':0, 'V':0, 'K':0, 'I':0, 'S':0, 'D':0, 'F':0, 'R':0, 'N':0, 'T':0, 'M':0, 'Q':0, 'Y':0, 'C':0, 'H':0, 'W':0}
    amino_acid_freq = {'A':0, 'G':0, 'L':0, 'P':0, 'E':0, 'V':0, 'K':0, 'I':0, 'S':0, 'D':0, 'F':0, 'R':0, 'N':0, 'T':0, 'M':0, 'Q':0, 'Y':0, 'C':0, 'H':0, 'W':0}

    for peptide in peptides:
        for amino_acid in peptide:
            amino_acid_num[amino_acid] = amino_acid_num.get(amino_acid, 0) + 1

    total_peptides = sum(amino_acid_num.values())
    for amino_acid, count in amino_acid_num.items():
        amino_acid_freq[amino_acid] = round((count / total_peptides), 4)

    return amino_acid_freq

train_freq = calculate_amino_acid_frequency(train_data)

weight_matrix = torch.zeros(24)
aa_matrix = torch.ones(20)
aa_matrix /= torch.as_tensor(list(train_freq.values()))
aa_matrix /= torch.sum(aa_matrix) + aa_matrix.mean()
eos_weight = aa_matrix.mean() / (torch.sum(aa_matrix) + aa_matrix.mean())
weight_matrix[2] = eos_weight
weight_matrix[4:] = aa_matrix


train_dataset = RecDataset(train_data, seq2idx)
test_dataset = RecDataset(test_data, seq2idx)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True, drop_last=True)

In [ ]:
# Positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len):
        super().__init__()
        pe = torch.zeros(max_len, embed_size)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2) * -(np.log(10000.0) / embed_size))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Multi-head attention module
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super().__init__()
        assert embed_size % num_heads == 0, "Embed size must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads

        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)
        self.scale = torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

    def forward(self, query, key, value, mask=None):
        N = query.size(0)
        Q = self.query(query).view(N, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(key).view(N, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(value).view(N, -1, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attention = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)
        out = out.transpose(1, 2).contiguous().view(N, -1, self.num_heads * self.head_dim)
        return self.fc_out(out)

# Add & Normalize
class AddNorm(nn.Module):
    def __init__(self, embed_size, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(embed_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return self.norm(x + self.dropout(sublayer))

# 前馈神经网络模块
class FeedForward(nn.Module):
    def __init__(self, embed_size, hidden_dim, dropout):
        super().__init__()
        self.fc1 = nn.Linear(embed_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

# Encoder
class EncoderLayer(nn.Module):
    def __init__(self, embed_size, num_heads, hidden_dim, dropout):
        super().__init__()
        self.self_attention = MultiHeadAttention(embed_size, num_heads)
        self.add_norm1 = AddNorm(embed_size, dropout)
        self.feed_forward = FeedForward(embed_size, hidden_dim, dropout)
        self.add_norm2 = AddNorm(embed_size, dropout)

    def forward(self, x, mask):
        attn_output = self.self_attention(x, x, x, mask)
        x = self.add_norm1(x, attn_output)
        ff_output = self.feed_forward(x)
        return self.add_norm2(x, ff_output)

class Transformer_Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers, num_heads, hidden_dim, max_len, dropout, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.positional_encoding = PositionalEncoding(embed_size, max_len)
        self.layers = nn.ModuleList([
            EncoderLayer(embed_size, num_heads, hidden_dim, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask):
        x = self.positional_encoding(self.embed(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x #[batch_size, seq_len, embedding_dim]

In [ ]:
class GRU_emb(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, max_seq_len, oracle_init=False):
        super(GRU_emb, self).__init__()
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len
        self.vocab_size = vocab_size

        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.gru2out = nn.Linear(2 * hidden_dim, embedding_dim)  

        if oracle_init:
            for p in self.parameters():
                nn.init.normal_(p, 0, 1)

    def init_hidden(self, batch_size=1):
        h = autograd.Variable(torch.zeros(2, batch_size, self.hidden_dim, device=device))
        return h

    def forward(self, inp, hidden):
        emb = self.embeddings(inp)  # [batch_size, seq_len, embedding_dim]
        out, hidden = self.gru(emb, hidden)  # out: [batch_size, seq_len, 2 * hidden_dim]
        out = self.gru2out(out)  # [batch_size, seq_len, embedding_dim]
        return out, hidden


In [ ]:
class WeightedFusion(nn.Module):
    def __init__(self, feature_dim):
        super(WeightedFusion, self).__init__()
        self.alpha = nn.Parameter(torch.tensor(0.5)) 
        self.linear = nn.Linear(feature_dim, feature_dim)

    def forward(self, x, out):
    
        alpha = torch.sigmoid(self.alpha) 

        fused = alpha * x + (1 - alpha) * out
        fused = self.linear(fused)
        return fused

In [ ]:
class TIGDM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, num_layers, num_heads, max_len, dropout, pad_idx, batch_size) -> None:
        super(TIGDM, self).__init__()
        self.emb = nn.Embedding(24, 512)
        # Encoder
        self.encoder = nn.GRU(embedding_dim, 256, num_layers=2, bidirectional=False, batch_first=True)
        self.hidden_to_latent = nn.Linear(256, 512)
        # Decoder
        self.decoder = nn.GRU(512, 512, bidirectional=False, batch_first=True)
        self.fc = nn.Sequential( 
            nn.LayerNorm([512]),
            nn.ReLU(),
            nn.Linear(512, 24))
        # module
        self.transformer_emb = Transformer_Encoder(vocab_size, embedding_dim, num_layers, num_heads, hidden_dim, max_len, dropout, pad_idx)
        self.gru_emb = GRU_emb(embedding_dim, hidden_dim, vocab_size, max_seq_len=27)
        self.feature_concat = WeightedFusion(embedding_dim)
        # parameter
        self.batch_size = batch_size
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim
        self.weight_param = nn.Parameter(torch.ones(1))
    
    def encoder_output(self, output):
        """
        Encode input sequence to latent space z.
        """
        _, h = self.encoder(output)
        # h = torch.cat((h[-2,:,:], h[-1,:,:]), 1) 
        h = h[-1, :, :]
        z = self.hidden_to_latent(h)
        return z
    
    def decoder_output(self, h, batch_seq):
        """
        Decode latent variable z to reconstruct sequence with weighted hidden states.
        """
        batch, seqlen = batch_seq.shape[0], batch_seq.shape[1]
        token = batch_seq[:, 0]
        logit = []
        h_list = [torch.zeros_like(h) for _ in range(seqlen)]  
        h_list[0] = h.squeeze(0)

        for t in range(seqlen - 1):
     
            weights = F.softmax(self.weight_param * torch.arange(len(h_list), device=h.device), dim=0)
            h_total = sum(w * h_list[idx] for idx, w in enumerate(weights))

     
            token_logit, h_out, token = self.onestep(token, h_total.unsqueeze(0))

    
            h_list[t+1] = h_out.squeeze(0)

       
            logit.append(token_logit.squeeze(dim=-2))

        logit = torch.stack(logit, dim=1)  
        return logit # [batch_size, seq_len - 1, vocab_size]
    
    def onestep(self, token, hidden):
        """
        Perform one decoding step.
        """
        token_embedding = self.emb(token.unsqueeze(-1))
        output, h = self.decoder(token_embedding, hidden)
        token_logit = F.softmax(self.fc(output), dim=-1)
        token_next = token_logit.squeeze(1).multinomial(1).squeeze(-1)
        return token_logit, h, token_next
    
    def make_src_mask(self, src):
        return (src != self.pad_idx).unsqueeze(1).unsqueeze(2)
    
    def init_hidden(self, batch_size=1):
        h = autograd.Variable(torch.zeros(2, batch_size, self.hidden_dim, device=device))
        return h
    
    def forward(self, x):
        # transformer
        src = x.to(device)
        src_mask = self.make_src_mask(src)
        out_transformer = self.transformer_emb(x, src_mask)
        # GRU
        h = self.init_hidden(self.batch_size)
        out_gru, _ = self.gru_emb(x, h)
        fused = self.feature_concat(out_transformer, out_gru)
        z = self.encoder_output(fused)
        logit = self.decoder_output(z, x)
        return z, logit

In [ ]:
embedding_dim = 256
hidden_dim = 512
vocab_size = len(vocab)
num_layers = 6
num_heads = 8
max_len = 27
dropout = 0.1
pad_idx = vocab["<pad>"]
model = WAE_model(embedding_dim, hidden_dim, vocab_size, num_layers, num_heads, max_len, dropout, pad_idx, batch_size=32).to(device)
model.load_state_dict(torch.load('/data/home/weng501/Users/kjh/model/TGW_model.pth'))
loss_fn = nn.CrossEntropyLoss(weight=weight_matrix).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:

best_test_loss = float("inf")
best_model_path = "TIGDM_tip_model.pth"
scheduler_TF = False
gradient_clip_TF = False

epoch = 1500
final_accuracy = 0

if scheduler_TF:
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

for i in range(epoch):
    total_train_loss = 0
    total_train_mmd = 0
    total_train_accuracy = 0
    train_token_num = 0
    model.train()
    
    for seq_ids in train_dataloader:
        target = seq_ids[:, 1:].to(device)
        seq_ids = seq_ids.to(device)
        z, output = model(seq_ids)  
        recon_loss = loss_fn(output.view(-1, output.size(2)), target.view(-1))
        z_prior = torch.randn_like(z).to(device)  
        mmd_loss = torch.mean((z - z_prior) ** 2)  

        loss = recon_loss + mmd_loss * 0.01
   
        mask = (target != 0)
        token_num = mask.long().sum().item()
        train_token_num += token_num

        accuracy = (output.argmax(-1)[mask] == target[mask]).sum()
        total_train_accuracy += accuracy.item()

        optimizer.zero_grad()
        loss.backward()


        if gradient_clip_TF:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  
        
        optimizer.step()

        total_train_loss += recon_loss.item()
        total_train_mmd += mmd_loss.item()

    train_acc = total_train_accuracy / train_token_num
    train_recon_loss = total_train_loss / len(train_dataloader)
    train_mmd_loss = total_train_mmd / len(train_dataloader)
    
    # print(f'Epoch: {i+1}, Train ACC: {train_acc:.4f}, recon_loss: {train_recon_loss:.4f}, mmd_loss: {train_mmd_loss:.4f}')

    total_test_loss = 0
    total_test_mmd = 0
    total_accuracy = 0
    test_token_num = 0
    model.eval()
    with torch.no_grad():
        for seq_ids in test_dataloader:
            target = seq_ids[:, 1:].to(device)
            seq_ids = seq_ids.to(device)

            z, output = model(seq_ids)  

            recon_loss = loss_fn(output.view(-1, output.size(2)), target.view(-1))

            z_prior = torch.randn_like(z).to(device)  

            loss = recon_loss + mmd_loss * 0.01
            
      
            mask = (target != 0)
            token_num = mask.long().sum().item()
            test_token_num += token_num

            accuracy = (output.argmax(-1)[mask] == target[mask]).sum()
            total_accuracy += accuracy.item()

            total_test_loss += recon_loss.item()
            total_test_mmd += mmd_loss.item()

        test_acc = total_accuracy / test_token_num
        test_recon_loss = total_test_loss / len(test_dataloader)
        test_mmd_loss = total_test_mmd / len(test_dataloader)
        
        print(f'Epoch: {i+1}, Test ACC: {test_acc:.4f}, recon_loss: {test_recon_loss:.4f}, mmd_loss: {test_mmd_loss:.4f}')

    if scheduler_TF:
        scheduler.step()

    if test_recon_loss < best_test_loss:
        best_test_loss = test_recon_loss
        torch.save(model.state_dict(), best_model_path)